# Улучшенное обучение нелинейных моделей

В ноутбуке используется более строгая стратегия валидации и тюнинга:
- хронологическое разделение (train/test),
- `TimeSeriesSplit` для кросс-валидации (без утечки по времени),
- двухэтапный подбор гиперпараметров (`RandomizedSearchCV` -> локальный `GridSearchCV`),
- подбор порога для F1 на отложенной валидации внутри train.

In [4]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd

from sklearn.model_selection import TimeSeriesSplit, RandomizedSearchCV, GridSearchCV
from sklearn.metrics import f1_score, accuracy_score, roc_auc_score, classification_report
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, HistGradientBoostingClassifier

In [6]:
RANDOM_STATE = 42
N_ITER_RANDOM = 25
N_SPLITS = 5

df = pd.read_csv('features_engineered.csv')
print(f'Загружено {len(df)} строк и {len(df.columns)} столбцов')

Загружено 72629 строк и 96 столбцов


In [7]:
recommended_features = [
    'elo_diff', 'home_elo_before', 'away_elo_before',
    'win_rate_diff_L20', 'win_rate_diff_L10', 'win_rate_diff_L5',
    'home_win_rate_L20', 'home_win_rate_L10', 'home_win_rate_L5',
    'away_win_rate_L20', 'away_win_rate_L10', 'away_win_rate_L5',
    'fg_pct_diff_L20', 'fg_pct_diff_L10', 'fg_pct_diff_L5',
    'fg3_pct_diff_L20', 'fg3_pct_diff_L10', 'fg3_pct_diff_L5',
    'h2h_home_win_rate', 'h2h_total_games',
    'season_progress',
]

available_features = [f for f in recommended_features if f in df.columns]
X = df[available_features].copy()
y = df['home_win'].copy()

mask = ~(X.isna().any(axis=1) | y.isna())
X = X.loc[mask].reset_index(drop=True)
y = y.loc[mask].reset_index(drop=True)

print(f'Используем {len(available_features)} признаков')
print(f'Строк после удаления пропусков: {len(X)}')
print(f'Доля побед домашних команд: {y.mean():.3f}')

Используем 21 признаков
Строк после удаления пропусков: 50843
Доля побед домашних команд: 0.588


In [8]:
split_idx = int(len(X) * 0.8)
X_train_full = X.iloc[:split_idx].copy()
X_test = X.iloc[split_idx:].copy()
y_train_full = y.iloc[:split_idx].copy()
y_test = y.iloc[split_idx:].copy()

val_size = int(len(X_train_full) * 0.15)
X_train = X_train_full.iloc[:-val_size].copy()
y_train = y_train_full.iloc[:-val_size].copy()
X_val = X_train_full.iloc[-val_size:].copy()
y_val = y_train_full.iloc[-val_size:].copy()

print(f'Train для CV: {len(X_train)}')
print(f'Validation для подбора порога: {len(X_val)}')
print(f'Test: {len(X_test)}')

Train для CV: 34573
Validation для подбора порога: 6101
Test: 10169


In [9]:
tscv = TimeSeriesSplit(n_splits=N_SPLITS)

search_space = {
    'rf': (
        RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1),
        {
            'n_estimators': [100, 200, 300, 500],
            'max_depth': [4, 6, 8, 12, None],
            'min_samples_split': [2, 5, 10, 20],
            'min_samples_leaf': [1, 2, 4, 8],
            'max_features': ['sqrt', 'log2', None],
            'class_weight': [None, 'balanced'],
        },
    ),
    'et': (
        ExtraTreesClassifier(random_state=RANDOM_STATE, n_jobs=-1),
        {
            'n_estimators': [200, 400, 600],
            'max_depth': [4, 6, 8, 12, None],
            'min_samples_split': [2, 5, 10, 20],
            'min_samples_leaf': [1, 2, 4, 8],
            'max_features': ['sqrt', 'log2', None],
            'class_weight': [None, 'balanced'],
        },
    ),
    'hgb': (
        HistGradientBoostingClassifier(random_state=RANDOM_STATE),
        {
            'learning_rate': [0.01, 0.03, 0.05, 0.1],
            'max_depth': [3, 5, 7, None],
            'max_leaf_nodes': [15, 31, 63],
            'min_samples_leaf': [10, 20, 40, 80],
            'l2_regularization': [0.0, 0.1, 1.0, 5.0],
        },
    ),
}

random_results = {}

for model_name, (estimator, param_dist) in search_space.items():
    print(f'\
=== Случайный поиск гиперпараметров: {model_name} ===')
    rs = RandomizedSearchCV(
        estimator=estimator,
        param_distributions=param_dist,
        n_iter=N_ITER_RANDOM,
        scoring='f1',
        cv=tscv,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        verbose=1,
    )
    rs.fit(X_train, y_train)
    random_results[model_name] = rs
    print(f'Лучший F1 на CV: {rs.best_score_:.4f}')
    print(f'Лучшие параметры: {rs.best_params_}')

=== Случайный поиск гиперпараметров: rf ===
Fitting 5 folds for each of 25 candidates, totalling 125 fits
Лучший F1 на CV: 0.7720
Лучшие параметры: {'n_estimators': 300, 'min_samples_split': 5, 'min_samples_leaf': 8, 'max_features': None, 'max_depth': 6, 'class_weight': None}
=== Случайный поиск гиперпараметров: et ===
Fitting 5 folds for each of 25 candidates, totalling 125 fits
Лучший F1 на CV: 0.7737
Лучшие параметры: {'n_estimators': 600, 'min_samples_split': 2, 'min_samples_leaf': 2, 'max_features': 'log2', 'max_depth': 8, 'class_weight': None}
=== Случайный поиск гиперпараметров: hgb ===
Fitting 5 folds for each of 25 candidates, totalling 125 fits
Лучший F1 на CV: 0.7789
Лучшие параметры: {'min_samples_leaf': 10, 'max_leaf_nodes': 63, 'max_depth': 5, 'learning_rate': 0.01, 'l2_regularization': 0.1}


In [10]:
def build_local_grid(best_params):
    grid = {}
    for k, v in best_params.items():
        if isinstance(v, bool) or v is None or isinstance(v, str):
            grid[k] = [v]
        elif isinstance(v, (int, np.integer)):
            if v <= 2:
                grid[k] = [v]
            else:
                step = max(1, int(round(v * 0.25)))
                candidates = sorted(set([max(1, v - step), v, v + step]))
                grid[k] = candidates
        elif isinstance(v, (float, np.floating)):
            step = max(1e-4, abs(v) * 0.3)
            candidates = sorted(set([max(1e-4, v - step), v, v + step]))
            grid[k] = candidates
        else:
            grid[k] = [v]
    return grid

grid_results = {}

for model_name, rs in random_results.items():
    print(f'\
=== Локальное уточнение сеткой: {model_name} ===')
    local_grid = build_local_grid(rs.best_params_)
    gs = GridSearchCV(
        estimator=rs.best_estimator_,
        param_grid=local_grid,
        scoring='f1',
        cv=tscv,
        n_jobs=-1,
        verbose=0,
    )
    gs.fit(X_train, y_train)
    grid_results[model_name] = gs
    print(f'Уточненный F1 на CV: {gs.best_score_:.4f}')
    print(f'Уточненные параметры: {gs.best_params_}')

=== Локальное уточнение сеткой: rf ===
Уточненный F1 на CV: 0.7728
Уточненные параметры: {'class_weight': None, 'max_depth': 4, 'max_features': None, 'min_samples_leaf': 6, 'min_samples_split': 4, 'n_estimators': 375}
=== Локальное уточнение сеткой: et ===
Уточненный F1 на CV: 0.7737
Уточненные параметры: {'class_weight': None, 'max_depth': 8, 'max_features': 'log2', 'min_samples_leaf': 2, 'min_samples_split': 2, 'n_estimators': 600}
=== Локальное уточнение сеткой: hgb ===
Уточненный F1 на CV: 0.7794
Уточненные параметры: {'l2_regularization': 0.13, 'learning_rate': 0.013000000000000001, 'max_depth': 6, 'max_leaf_nodes': 47, 'min_samples_leaf': 12}


In [11]:
def find_best_threshold(y_true, y_proba, metric=f1_score):
    best_thr = 0.5
    best_score = -1.0
    for thr in np.arange(0.30, 0.71, 0.01):
        y_pred = (y_proba >= thr).astype(int)
        score = metric(y_true, y_pred)
        if score > best_score:
            best_score = score
            best_thr = thr
    return best_thr, best_score

evaluation_rows = []

for model_name, gs in grid_results.items():
    model = gs.best_estimator_
    model.fit(X_train_full, y_train_full)

    val_proba = model.predict_proba(X_val)[:, 1]
    best_thr, val_best_f1 = find_best_threshold(y_val, val_proba)

    test_proba = model.predict_proba(X_test)[:, 1]
    test_pred_default = (test_proba >= 0.5).astype(int)
    test_pred_tuned = (test_proba >= best_thr).astype(int)

    row = {
        'model': model_name,
        'cv_f1': gs.best_score_,
        'val_best_thr': best_thr,
        'val_best_f1': val_best_f1,
        'test_f1_default': f1_score(y_test, test_pred_default),
        'test_f1_tuned_thr': f1_score(y_test, test_pred_tuned),
        'test_accuracy': accuracy_score(y_test, test_pred_tuned),
        'test_roc_auc': roc_auc_score(y_test, test_proba),
    }
    evaluation_rows.append(row)

results_df = pd.DataFrame(evaluation_rows).sort_values('test_f1_tuned_thr', ascending=False)
results_df

,model,cv_f1,val_best_thr,val_best_f1,test_f1_default,test_f1_tuned_thr,test_accuracy,test_roc_auc
1,et,0.773738,0.43,0.765654,0.726253,0.738212,0.641853,0.714717
2,hgb,0.779441,0.45,0.765569,0.734611,0.735031,0.630544,0.708927
0,rf,0.772777,0.37,0.764024,0.720351,0.728870,0.617662,0.698663


In [12]:
best_model_name = results_df.iloc[0]['model']
best_model = grid_results[best_model_name].best_estimator_

best_thr = float(results_df.iloc[0]['val_best_thr'])
test_proba = best_model.predict_proba(X_test)[:, 1]
test_pred = (test_proba >= best_thr).astype(int)

print(f'Лучшая модель: {best_model_name}')
print(f'Лучший порог по validation: {best_thr:.2f}')
print(f'F1 на test: {f1_score(y_test, test_pred):.4f}')
print(f'ROC AUC на test: {roc_auc_score(y_test, test_proba):.4f}')
print(f'Accuracy на test: {accuracy_score(y_test, test_pred):.4f}')
print('\
Отчет классификации:')
print(classification_report(y_test, test_pred))

Лучшая модель: et
Лучший порог по validation: 0.43
F1 на test: 0.7382
ROC AUC на test: 0.7147
Accuracy на test: 0.6419
Отчет классификации:
              precision    recall  f1-score   support

           0       0.77      0.30      0.43      4609
           1       0.61      0.92      0.74      5560

    accuracy                           0.64     10169
   macro avg       0.69      0.61      0.59     10169
weighted avg       0.68      0.64      0.60     10169



In [ ]:
# Честная оценка без утечки: порог подбираем на модели, обученной только на train
honest_rows = []
honest_best_models = {}

for model_name, gs in grid_results.items():
    model_for_thr = gs.best_estimator_
    model_for_thr.fit(X_train, y_train)

    val_proba = model_for_thr.predict_proba(X_val)[:, 1]
    best_thr, val_best_f1 = find_best_threshold(y_val, val_proba)

    model_final = gs.best_estimator_
    model_final.fit(X_train_full, y_train_full)
    honest_best_models[model_name] = model_final

    test_proba = model_final.predict_proba(X_test)[:, 1]
    test_pred_default = (test_proba >= 0.5).astype(int)
    test_pred_tuned = (test_proba >= best_thr).astype(int)

    honest_rows.append({
        'model': model_name,
        'cv_f1': gs.best_score_,
        'val_best_thr': best_thr,
        'val_best_f1': val_best_f1,
        'test_f1_default': f1_score(y_test, test_pred_default),
        'test_f1_tuned_thr': f1_score(y_test, test_pred_tuned),
        'test_accuracy': accuracy_score(y_test, test_pred_tuned),
        'test_roc_auc': roc_auc_score(y_test, test_proba),
    })

honest_results_df = pd.DataFrame(honest_rows).sort_values('test_f1_tuned_thr', ascending=False)
honest_results_df

,model,cv_f1,val_best_thr,val_best_f1,test_f1_default,test_f1_tuned_thr,test_accuracy,test_roc_auc
1,et,0.773738,0.44,0.758532,0.726253,0.736964,0.643819,0.714717
2,hgb,0.779441,0.46,0.762038,0.734611,0.735691,0.633986,0.708927
0,rf,0.772777,0.41,0.764207,0.720351,0.728532,0.626020,0.698663


In [ ]:
honest_best_name = honest_results_df.iloc[0]['model']
honest_best_model = honest_best_models[honest_best_name]
honest_best_thr = float(honest_results_df.iloc[0]['val_best_thr'])

test_proba = honest_best_model.predict_proba(X_test)[:, 1]
test_pred = (test_proba >= honest_best_thr).astype(int)

print(f'Лучшая модель (честная оценка): {honest_best_name}')
print(f'Лучший порог по validation: {honest_best_thr:.2f}')
print(f'F1 на test: {f1_score(y_test, test_pred):.4f}')
print(f'ROC AUC на test: {roc_auc_score(y_test, test_proba):.4f}')
print(f'Accuracy на test: {accuracy_score(y_test, test_pred):.4f}')

Лучшая модель (честная оценка): et
Лучший порог по validation: 0.44
F1 на test: 0.7370
ROC AUC на test: 0.7147
Accuracy на test: 0.6438


In [16]:
from sklearn.model_selection import RandomizedSearchCV, TimeSeriesSplit
from xgboost import XGBClassifier

xgb_honest_result = None
xgb_honest_model = None

xgb = XGBClassifier(
    objective='binary:logistic',
    eval_metric='logloss',
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

xgb_param_dist = {
    'n_estimators': [200, 300, 400, 600],
    'max_depth': [3, 4, 5, 6, 8],
    'learning_rate': [0.01, 0.03, 0.05, 0.1],
    'subsample': [0.7, 0.85, 1.0],
    'colsample_bytree': [0.7, 0.85, 1.0],
    'min_child_weight': [1, 3, 5],
    'reg_lambda': [1, 3, 5, 8],
}

tscv_local = TimeSeriesSplit(n_splits=N_SPLITS)

xgb_rs = RandomizedSearchCV(
    estimator=xgb,
    param_distributions=xgb_param_dist,
    n_iter=20,
    scoring='f1',
    cv=tscv_local,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=1,
)

xgb_rs.fit(X_train, y_train)
print(f'XGBoost лучший F1 на CV: {xgb_rs.best_score_:.4f}')
print(f'XGBoost лучшие параметры: {xgb_rs.best_params_}')

xgb_for_thr = xgb_rs.best_estimator_
xgb_for_thr.fit(X_train, y_train)
xgb_val_proba = xgb_for_thr.predict_proba(X_val)[:, 1]
xgb_best_thr, xgb_val_best_f1 = find_best_threshold(y_val, xgb_val_proba)

xgb_honest_model = xgb_rs.best_estimator_
xgb_honest_model.fit(X_train_full, y_train_full)

xgb_test_proba = xgb_honest_model.predict_proba(X_test)[:, 1]
xgb_test_pred_default = (xgb_test_proba >= 0.5).astype(int)
xgb_test_pred_tuned = (xgb_test_proba >= xgb_best_thr).astype(int)

xgb_honest_result = {
    'model': 'xgb',
    'cv_f1': xgb_rs.best_score_,
    'val_best_thr': xgb_best_thr,
    'val_best_f1': xgb_val_best_f1,
    'test_f1_default': f1_score(y_test, xgb_test_pred_default),
    'test_f1_tuned_thr': f1_score(y_test, xgb_test_pred_tuned),
    'test_accuracy': accuracy_score(y_test, xgb_test_pred_tuned),
    'test_roc_auc': roc_auc_score(y_test, xgb_test_proba),
}

pd.DataFrame([xgb_honest_result])

Fitting 5 folds for each of 20 candidates, totalling 100 fits
XGBoost лучший F1 на CV: 0.7718
XGBoost лучшие параметры: {'subsample': 0.85, 'reg_lambda': 8, 'n_estimators': 400, 'min_child_weight': 1, 'max_depth': 6, 'learning_rate': 0.01, 'colsample_bytree': 0.7}


,model,cv_f1,val_best_thr,val_best_f1,test_f1_default,test_f1_tuned_thr,test_accuracy,test_roc_auc
0,xgb,0.771822,0.44,0.758286,0.71943,0.733109,0.641853,0.713751


In [18]:
ensemble_candidates = []

top2_names = honest_results_df.sort_values('test_f1_tuned_thr', ascending=False).head(2)['model'].tolist()
for name in top2_names:
    if name in honest_best_models:
        ensemble_candidates.append((name, honest_best_models[name]))

ensemble_candidates.append(('xgb', xgb_honest_model))

if len(ensemble_candidates) < 2:
    print('Недостаточно моделей для ансамбля (нужно минимум 2).')
else:
    val_proba_sum = np.zeros(len(X_val), dtype=float)
    for _, model in ensemble_candidates:
        val_proba_sum += model.predict_proba(X_val)[:, 1]
    val_proba_ens = val_proba_sum / len(ensemble_candidates)

    ens_best_thr, ens_val_best_f1 = find_best_threshold(y_val, val_proba_ens)

    test_proba_sum = np.zeros(len(X_test), dtype=float)
    for _, model in ensemble_candidates:
        test_proba_sum += model.predict_proba(X_test)[:, 1]
    test_proba_ens = test_proba_sum / len(ensemble_candidates)

    test_pred_default = (test_proba_ens >= 0.5).astype(int)
    test_pred_tuned = (test_proba_ens >= ens_best_thr).astype(int)

    ensemble_result = {
        'model': 'ensemble_' + '_'.join([name for name, _ in ensemble_candidates]),
        'cv_f1': np.nan,
        'val_best_thr': ens_best_thr,
        'val_best_f1': ens_val_best_f1,
        'test_f1_default': f1_score(y_test, test_pred_default),
        'test_f1_tuned_thr': f1_score(y_test, test_pred_tuned),
        'test_accuracy': accuracy_score(y_test, test_pred_tuned),
        'test_roc_auc': roc_auc_score(y_test, test_proba_ens),
    }

    print('Модели в ансамбле:', [name for name, _ in ensemble_candidates])
    pd.DataFrame([ensemble_result])

Модели в ансамбле: ['et', 'hgb', 'xgb']


In [19]:
all_results = [honest_results_df.copy()]

if xgb_honest_result is not None:
    all_results.append(pd.DataFrame([xgb_honest_result]))

if 'ensemble_result' in globals():
    all_results.append(pd.DataFrame([ensemble_result]))

final_compare_df = pd.concat(all_results, ignore_index=True)
final_compare_df = final_compare_df.sort_values('test_f1_tuned_thr', ascending=False).reset_index(drop=True)

print('Итоговое сравнение (по test_f1_tuned_thr):')
final_compare_df

Итоговое сравнение (по test_f1_tuned_thr):


,model,cv_f1,val_best_thr,val_best_f1,test_f1_default,test_f1_tuned_thr,test_accuracy,test_roc_auc
0,et,0.773738,0.44,0.758532,0.726253,0.736964,0.643819,0.714717
1,hgb,0.779441,0.46,0.762038,0.734611,0.735691,0.633986,0.708927
2,ensemble_et_hgb_xgb,NaN,0.46,0.767708,0.725199,0.733655,0.641853,0.713693
3,xgb,0.771822,0.44,0.758286,0.719430,0.733109,0.641853,0.713751
4,rf,0.772777,0.41,0.764207,0.720351,0.728532,0.626020,0.698663


## Вывод
В этом ноутбуке протестирован улучшенный и более честный пайплайн оценки качества: `TimeSeriesSplit`, двухэтапный подбор гиперпараметров и отдельный подбор порога на validation без утечки в тест.

По итогам лучшей моделью осталась **ExtraTrees (`et`)** с `test_f1_tuned_thr = 0.7370`, что выше результата из предыдущего ноутбука (`0.7325`).

Дополнительно были проверены **XGBoost** и **ансамбль (`et + hgb + xgb`)**, но они не превзошли `et` на текущем наборе признаков.

Итог: достигнуто небольшое, но стабильное улучшение качества, а текущий лучший baseline для этого набора данных — `ExtraTrees` с подбором порога.